## _Scrapper_ | Site Anvisa de Registro e Notificações de Medicamentos
### Definição de Variáveis

In [29]:
from pathlib import Path

DATASET_DIR = 'samples/dataset'
DATASET_FILENAME = 'Anvisa_REGISTRO_MEDICAMENTOS.csv'

ROOT_DIR = Path().resolve().__str__()
DATASET_PATH = ROOT_DIR + '/' + DATASET_DIR
Path.mkdir(Path(DATASET_PATH), parents=True, exist_ok=True)
DATASET_FILEPATH = Path(DATASET_PATH + '/' + DATASET_FILENAME)
TOTAL_ELEMENTS = 50000

### Definição da request

In [30]:
url = "https://consultas.anvisa.gov.br/api/consulta/medicamento/produtos/"

In [31]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:152.0) Gecko/20100101 Firefox/152.0",
    "Accept": "application/json, text/plain, */*",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept-Encoding": "gzip, deflate, br, zstd",
    "If-Modified-Since": "Mon, 26 Jul 1997 05:00:00 GMT",
    "Cache-Control": "no-cache",
    "Pragma": "no-cache",
    "Authorization": "Guest",
    "x-dtpc": "4$266473457_314h85vLUAVIHKFRGMWAOEQDLOVUGPFNFINOJVM-0e0",
    "x-dtreferer": "https://consultas.anvisa.gov.br/",
    "Connection": "keep-alive",
    "Referer": "https://consultas.anvisa.gov.br/",
    "Sec-Fetch-Dest": "empty",
    "Sec-Fetch-Mode": "cors",
    "Sec-Fetch-Site": "same-origin",
    "Priority": "u=0",
}

In [32]:
params = [
    {
        "column": "",
        "count": TOTAL_ELEMENTS,
        "filter[checkNotificado]": "true", # Notificado
        "filter[checkRegistrado]": "true",
        "filter[situacaoRegistro]": "V", # Ativo
        "order": "asc",
        "page": 1,
    },
    {
        "column": "",
         "count": TOTAL_ELEMENTS,
         "filter[checkNotificado]": "true", # Notificado
         "filter[checkRegistrado]": "true",
         "filter[situacaoRegistro]": "C", # Inativo
         "order": "asc",
         "page": 1,
     },
]

### Execução da Request

In [33]:
import cloudscraper as cs

try:
    scrapper = cs.create_scraper()
    responses = [scrapper.get(url, params=param, headers=headers) for param in params]

    print(f'Raw data retrieved from {url}')

except cs.exceptions.CaptchaException as e:
    print(e)

Raw data retrieved from https://consultas.anvisa.gov.br/api/consulta/medicamento/produtos/


### Tratamento dos dados

In [35]:
codigo = []
medicacao = []
dataVencimentoRegistro = []
situacaoApresentacao = []
categoriaRegulatoriaDescricao = []
categoriaRegulatoriaCategoria = []
tipoAutorizacao = []
razaoSocial = []
cnpjFormatado = []
numeroProcessoFormatado = []
url = []

In [36]:
import json

for response_json in responses:
    response_json = json.loads(response_json.text)
    for item in response_json['content']:

        produto = dict(item['produto'])
        codigo.append(produto['codigo'])
        medicacao.append(produto['nome'])
        situacaoApresentacao.append(produto['situacaoApresentacao'])
        dataVencimentoRegistro.append(str(produto['dataVencimentoRegistro']).split('T')[0])
        categoriaRegulatoriaDescricao.append(produto['categoriaRegulatoria']['descricao'])
        categoriaRegulatoriaCategoria.append(produto['categoriaMedicamentoNotificado'])
        tipoAutorizacao.append(produto['tipoAutorizacao'])
        url.append(f'https://consultas.anvisa.gov.br/#/medicamentos/{produto['codigo']}')

        empresa = dict(item['empresa'])
        razaoSocial.append(empresa['razaoSocial'])
        cnpjFormatado.append(empresa['cnpjFormatado'])

        processo = dict(item['processo'])
        numeroProcessoFormatado.append(item['processo']['numeroProcessoFormatado'])


In [37]:
import pandas as pd

df = pd.DataFrame(data={
                    'codigo': codigo,
                    'medicamento': medicacao,
                    'situacao': situacaoApresentacao,
                    'data_Vencimento_Registro': dataVencimentoRegistro,
                    'categoria_Regulatoria_Descricao': categoriaRegulatoriaDescricao,
                    'tipo_Autorizacao': tipoAutorizacao,
                    'razao_Social': razaoSocial,
                    'cnpj_Formatado': cnpjFormatado,
                    'numero_Processo_Formatado': numeroProcessoFormatado,
                    'url': url,
                   })

### Exibição das Informações

In [38]:
docs = [row[1].to_markdown(tablefmt=None) for row in df.iterrows()]

In [39]:
print('\n'.join(docs[:5]))

                                 0
-------------------------------  ------------------------------------------------------
codigo                           1098085
medicamento                      sinvastatina
situacao                         Ativo
data_Vencimento_Registro         2036-04-01
categoria_Regulatoria_Descricao  Genérico
tipo_Autorizacao                 REGISTRADO
razao_Social                     NOVARTIS BIOCIENCIAS S.A
cnpj_Formatado                   56.994.502/0001-30
numero_Processo_Formatado        25351.916068/2016-29
url                              https://consultas.anvisa.gov.br/#/medicamentos/1098085
                                 1
-------------------------------  -----------------------------------------------------
codigo                           466454
medicamento                      AMATO
situacao                         Ativo
data_Vencimento_Registro         2026-11-01
categoria_Regulatoria_Descricao  Similar
tipo_Autorizacao                 REGISTRADO


### Exportação para CSV

In [40]:
df.to_csv(DATASET_FILEPATH, index=False, encoding='utf-8')

### Checksum

In [41]:
import hashlib
print(f'{DATASET_FILEPATH.name}: {hashlib.md5(open(DATASET_FILEPATH, 'rb').read()).hexdigest()}')

Anvisa_REGISTRO_MEDICAMENTOS.csv: 6f6ec84bf01c4014432dd3047a4649e7
